In [1]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda
from langchain_core.runnables import RunnablePassthrough
from langchain_core.runnables import RunnableParallel

In [ ]:
def char_counts(text: str) -> str:
    print(f"Calculating character count for: {text}")
    return "Character count: " + str(len(text))

def word_counts(text: str) -> str:
    print(f"Calculating word count for: {text}")
    return "Word count: " + str(len(text.split()))

In [7]:
char_count_runnable = RunnableLambda(char_counts)
word_count_runnable = RunnableLambda(word_counts)

prompt = ChatPromptTemplate.from_template("""Explain these inputs: {input1} and {input2} in 2 sentences.""")
llm = ChatOllama(
    model = "qwen3",
    base_url="http://localhost:11434"
)

chain = prompt | llm | StrOutputParser() | {
    "char_count": char_count_runnable, 
    "word_count": word_count_runnable,
    "output": RunnablePassthrough()
    } 
output = chain.invoke({"input1": "Hello world", "input2": "How are you?"})
print(output)


{'char_count': 185, 'word_count': 30, 'output': '"Hello world" is a simple greeting often used in programming examples to test code execution. "How are you?" is a common question to inquire about someone\'s well-being or current state.'}


In [ ]:
#Another way to do the same thing is to use RunnableParallel which will run the char_count_runnable and word_count_runnable in parallel and return the results in a dictionary.

parallel = RunnableParallel({
    "char_count": char_count_runnable,
    "word_count": word_count_runnable,
    "output": RunnablePassthrough()
})
parallel_output = prompt |llm | StrOutputParser() | parallel
output = parallel_output.invoke({"input1": "Hello world", "input2": "How are you?"})
print(output)

{'char_count': 269, 'word_count': 41, 'output': '"Hello world" is a common greeting used to introduce oneself or test basic programming code, while "How are you?" is a friendly question to inquire about someone\'s well-being or current state. Both phrases serve as simple, conversational starters in different contexts.'}
